In [21]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import sentencepiece as spm

data = r"D:\ML\ak_GPT\data\processed_recipes.txt"



# DATASET

In [22]:
import sentencepiece as spm
import numpy as np

sp = spm.SentencePieceProcessor()
sp.load(r"D:\ML\ak_GPT\data\recipe_tokenizer.model")

lengths = []
with open(r"D:\ML\ak_GPT\data\processed_recipes.txt", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 10000:  # Sample first 10K recipes
            break
        tokens = sp.encode(line.strip())
        lengths.append(len(tokens))

print(f"Mean: {np.mean(lengths):.0f} tokens")
print(f"Median: {np.median(lengths):.0f}")
print(f"90th percentile: {np.percentile(lengths, 90):.0f}")
print(f"95th percentile: {np.percentile(lengths, 95):.0f}")
print(f"Max: {np.max(lengths)}")

Mean: 115 tokens
Median: 109
90th percentile: 167
95th percentile: 187
Max: 428


### Notes 
1. Block size is 256 as most recipes fall within that block size 

Next we're implementing the dataset class 

A custom Dataset class must implement three functions: __init__, __len__, and __getitem__. 

In [23]:
# Parameters 
block_size = 256 # This is the sequence length that the model will be trained on
batch_size = 64 # Number of sequences per batch

import os
import json

class RecipeDataset(Dataset): 
    def __init__(
        self,
        data_path,
        tokenizer,
        block_size,
        split="train",
        train_ratio=0.95,
        cache_tokens=None,
        cache_meta=None,
        force_rebuild=False,
    ):
        self.tokenizer = tokenizer
        self.block_size = block_size

        if split not in {"train", "test"}:
            raise ValueError("split must be 'train' or 'test'")
        if not (0.0 < train_ratio < 1.0):
            raise ValueError("train_ratio must be between 0 and 1")

        default_cache_tokens = data_path.replace(".txt", ".tokens.npy")
        default_cache_meta = data_path.replace(".txt", ".tokens.meta.json")
        self.cache_tokens = cache_tokens or default_cache_tokens
        self.cache_meta = cache_meta or default_cache_meta

        all_tokens = self._load_or_build_tokens(data_path, force_rebuild)
        self.total_tokens = len(all_tokens)

        split_idx = int(self.total_tokens * train_ratio)
        if split == "train":
            self.tokens = all_tokens[:split_idx]
        else:
            self.tokens = all_tokens[split_idx:]

        min_needed = self.block_size + 1
        if len(self.tokens) < min_needed:
            raise ValueError(
                f"Split '{split}' has {len(self.tokens)} tokens; need at least {min_needed}. "
                "Increase data size, lower block_size, or adjust train_ratio."
            )

    def _load_or_build_tokens(self, data_path, force_rebuild):
        if (not force_rebuild) and self._is_cache_valid(data_path):
            return np.load(self.cache_tokens, mmap_mode="r")

        all_tokens = []
        with open(data_path, "r", encoding="utf-8") as f:
            for line in f:
                tokens = self.tokenizer.encode(line.strip())
                all_tokens.extend(tokens)

        tokens = np.array(all_tokens, dtype=np.int32)
        self._save_cache(tokens, data_path)
        return tokens

    def _is_cache_valid(self, data_path):
        if not (os.path.exists(self.cache_tokens) and os.path.exists(self.cache_meta)):
            return False

        try:
            with open(self.cache_meta, "r", encoding="utf-8") as f:
                meta = json.load(f)
        except (OSError, json.JSONDecodeError):
            return False

        if not os.path.exists(data_path):
            return False

        checks = [
            meta.get("source_text") == data_path,
            meta.get("source_mtime") == os.path.getmtime(data_path),
            meta.get("source_size") == os.path.getsize(data_path),
            meta.get("tokenizer_vocab_size") == int(self.tokenizer.get_piece_size()),
            meta.get("dtype") == "int32",
        ]
        return all(checks)

    def _save_cache(self, tokens, data_path):
        np.save(self.cache_tokens, tokens)
        meta = {
            "dtype": str(tokens.dtype),
            "num_tokens": int(len(tokens)),
            "source_text": data_path,
            "source_mtime": os.path.getmtime(data_path),
            "source_size": os.path.getsize(data_path),
            "tokenizer_vocab_size": int(self.tokenizer.get_piece_size()),
        }
        with open(self.cache_meta, "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)

    def __len__(self):
        return (len(self.tokens) - 1) // self.block_size
    
    def __getitem__(self, idx):
        start = idx * self.block_size
        chunk = self.tokens[start : start + self.block_size + 1]  # +1!
        
        x = torch.tensor(chunk[:-1], dtype=torch.long)  # input
        y = torch.tensor(chunk[1:],  dtype=torch.long)  # target
        return x, y
    

In [24]:
train_dataset = RecipeDataset(data, sp, block_size, split="train")
test_dataset = RecipeDataset(data, sp, block_size, split="test")

print(f"Total tokens (full): {train_dataset.total_tokens:,}")
print(f"Train tokens: {len(train_dataset.tokens):,}")
print(f"Test tokens: {len(test_dataset.tokens):,}")
print(f"Train chunks: {len(train_dataset)}")
print(f"Test chunks: {len(test_dataset)}")

x, y = train_dataset[0]
print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")
print(f"x[:5]: {x[:5]}")
print(f"y[:5]: {y[:5]}")

# Verify y is x shifted by one
assert torch.all(x[1:] == y[:-1]), "Shift is wrong!"
print("Shift verified!")

Total tokens (full): 398,115,169
Train tokens: 378,209,410
Test tokens: 19,905,759
Train chunks: 1477380
Test chunks: 77756
x shape: torch.Size([256])
y shape: torch.Size([256])
x[:5]: tensor([31573,     4, 31573,     6,  2956])
y[:5]: tensor([    4, 31573,     6,  2956, 31607])
Shift verified!


## CACHE the Dataset

In [25]:
cache_tokens = data.replace(".txt", ".tokens.npy")
cache_meta = data.replace(".txt", ".tokens.meta.json")

print("Cache tokens file:", cache_tokens)
print("Cache meta file:", cache_meta)
print("Cache exists:", os.path.exists(cache_tokens) and os.path.exists(cache_meta))

# Use this only when you intentionally want to retokenize and refresh cache
# _ = RecipeDataset(data, sp, block_size, split="train", force_rebuild=True)

Cache tokens file: D:\ML\ak_GPT\data\processed_recipes.tokens.npy
Cache meta file: D:\ML\ak_GPT\data\processed_recipes.tokens.meta.json
Cache exists: True


# DATALOADER

In [27]:
# Create train/test dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

x_train, y_train = next(iter(train_loader))
print(f"train x_batch shape: {x_train.shape}")
print(f"train y_batch shape: {y_train.shape}")

x_test, y_test = next(iter(test_loader))
print(f"test x_batch shape: {x_test.shape}")
print(f"test y_batch shape: {y_test.shape}")

train x_batch shape: torch.Size([64, 256])
train y_batch shape: torch.Size([64, 256])
test x_batch shape: torch.Size([64, 256])
test y_batch shape: torch.Size([64, 256])


In [33]:
print(f"Number of train batches: {len(iter(train_loader))}")
print(f"Number of test batches: {len(iter(test_loader))}")

Number of train batches: 23085
Number of test batches: 1215
